# Fannie Mae — Stratified Subset Sampling
Builds a small but outcome-rich subset for RL training:
- 100% of loans that ever went delinquent
- 10% random sample of always-current loans

Output: a single `subset.parquet` file ready for RL environment construction.

In [1]:
import os
import tarfile
import pandas as pd
import numpy as np

## CONFIG

In [2]:
TAR_PATH       = "fannie_mae.tar.gz"
EXTRACT_DIR    = "data/fannie_mae"
OUTPUT_PATH    = "data/subset.parquet"
CURRENT_SAMPLE = 0.10   # Fraction of always-current loans to keep
RANDOM_SEED    = 42

os.makedirs("data", exist_ok=True)
print(f"Source : {TAR_PATH}")
print(f"Output : {OUTPUT_PATH}")
print(f"Current loan sample rate: {CURRENT_SAMPLE*100:.0f}%")

Source : fannie_mae.tar.gz
Output : data/subset.parquet
Current loan sample rate: 10%


## Step 1 — Extract tar.gz

In [3]:
parquet_files = sorted([
    f for f in os.listdir(EXTRACT_DIR) if f.endswith(".parquet")
]) if os.path.exists(EXTRACT_DIR) else []

if parquet_files:
    print(f"Already extracted — found {len(parquet_files)} parquet files, skipping")
else:
    print(f"Extracting {TAR_PATH}...")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(".")
    parquet_files = sorted([
        f for f in os.listdir(EXTRACT_DIR) if f.endswith(".parquet")
    ])
    print(f"Extracted {len(parquet_files)} parquet files")

print("\nQuarters available:")
for f in parquet_files:
    size_mb = os.path.getsize(os.path.join(EXTRACT_DIR, f)) / 1e6
    print(f"  {f:<25} {size_mb:>7.0f} MB")

Extracting fannie_mae.tar.gz...


FileNotFoundError: [Errno 2] No such file or directory: 'fannie_mae.tar.gz'

## Step 2 — Scan all quarters to find delinquent loan IDs
We pass through each file once, collecting any Loan Identifier that ever had delinquency > 0.

In [9]:
delinquent_ids = set()
all_ids        = set()

print("Scanning for delinquent loans...")
for fname in parquet_files:
    path = os.path.join(EXTRACT_DIR, fname)
    df = pd.read_parquet(path, columns=["Loan Identifier", "Current Loan Delinquency Status"])

    all_ids.update(df["Loan Identifier"].unique())

    # Delinquency status: 0 = current, anything else = delinquent
    # Cast to numeric — field can sometimes be string coded
    delinq = pd.to_numeric(df["Current Loan Delinquency Status"], errors="coerce")
    delinquent_ids.update(
        df.loc[delinq > 0, "Loan Identifier"].unique()
    )

    print(f"  {fname:<25} — running delinquent: {len(delinquent_ids):,}  total: {len(all_ids):,}")

current_ids = all_ids - delinquent_ids

print(f"\nTotal unique loans  : {len(all_ids):,}")
print(f"Ever delinquent     : {len(delinquent_ids):,} ({len(delinquent_ids)/len(all_ids)*100:.1f}%)")
print(f"Always current      : {len(current_ids):,} ({len(current_ids)/len(all_ids)*100:.1f}%)")

Scanning for delinquent loans...
  2022Q1.parquet            — running delinquent: 89,419  total: 793,818
  2022Q2.parquet            — running delinquent: 158,893  total: 1,361,431
  2022Q3.parquet            — running delinquent: 205,256  total: 1,739,344
  2022Q4.parquet            — running delinquent: 234,376  total: 2,013,006
  2023Q1.parquet            — running delinquent: 253,650  total: 2,223,816
  2023Q2.parquet            — running delinquent: 273,333  total: 2,500,530
  2023Q3.parquet            — running delinquent: 290,982  total: 2,767,258
  2023Q4.parquet            — running delinquent: 303,609  total: 2,983,192
  2024Q1.parquet            — running delinquent: 313,481  total: 3,171,694
  2024Q2.parquet            — running delinquent: 325,044  total: 3,426,342
  2024Q3.parquet            — running delinquent: 335,692  total: 3,703,622
  2024Q4.parquet            — running delinquent: 343,685  total: 3,954,168
  2025Q1.parquet            — running delinquent: 348,742 

## Step 3 — Sample current loan IDs

In [10]:
rng = np.random.default_rng(RANDOM_SEED)
current_ids_list  = list(current_ids)
n_sample          = int(len(current_ids_list) * CURRENT_SAMPLE)
sampled_current   = set(rng.choice(current_ids_list, size=n_sample, replace=False))

keep_ids = delinquent_ids | sampled_current

print(f"Keeping all {len(delinquent_ids):,} delinquent loans")
print(f"Keeping {len(sampled_current):,} of {len(current_ids):,} current loans ({CURRENT_SAMPLE*100:.0f}% sample)")
print(f"Total loans in subset: {len(keep_ids):,}")

Keeping all 355,363 delinquent loans
Keeping 430,153 of 4,301,536 current loans (10% sample)
Total loans in subset: 785,516


## Step 4 — Build subset from all quarters

In [11]:
chunks = []
total_rows = 0

print("Building subset...")
for fname in parquet_files:
    path = os.path.join(EXTRACT_DIR, fname)
    df   = pd.read_parquet(path)

    df_filtered = df[df["Loan Identifier"].isin(keep_ids)]
    chunks.append(df_filtered)
    total_rows += len(df_filtered)

    print(f"  {fname:<25} — kept {len(df_filtered):,} of {len(df):,} rows")

print(f"\nCombining {len(chunks)} quarters...")
df_subset = pd.concat(chunks, ignore_index=True)

# Sort by loan then time for clean episode iteration
df_subset = df_subset.sort_values(
    ["Loan Identifier", "Monthly Reporting Period"]
).reset_index(drop=True)

print(f"Final subset shape: {df_subset.shape}")

Building subset...
  2022Q1.parquet            — kept 6,650,334 of 32,734,380 rows
  2022Q2.parquet            — kept 4,628,148 of 21,846,208 rows
  2022Q3.parquet            — kept 2,849,967 of 13,375,587 rows
  2022Q4.parquet            — kept 1,756,920 of 8,867,388 rows
  2023Q1.parquet            — kept 1,152,059 of 6,213,667 rows
  2023Q2.parquet            — kept 1,240,189 of 7,503,887 rows
  2023Q3.parquet            — kept 1,044,731 of 6,492,480 rows
  2023Q4.parquet            — kept 698,468 of 4,497,693 rows
  2024Q1.parquet            — kept 522,418 of 3,523,746 rows
  2024Q2.parquet            — kept 580,580 of 4,100,715 rows
  2024Q3.parquet            — kept 511,232 of 3,781,288 rows
  2024Q4.parquet            — kept 350,287 of 2,724,362 rows
  2025Q1.parquet            — kept 189,625 of 1,523,689 rows
  2025Q2.parquet            — kept 141,506 of 1,210,504 rows
  2025Q3.parquet            — kept 57,835 of 532,607 rows

Combining 15 quarters...
Final subset shape: (22374

## Step 5 — Save

In [12]:
df_subset.to_parquet(OUTPUT_PATH, index=False, compression="snappy")
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved to {OUTPUT_PATH} ({size_mb:.0f} MB)")

Saved to data/subset.parquet (370 MB)


## Step 6 — Verify

In [13]:
df = pd.read_parquet(OUTPUT_PATH)

print(f"Shape            : {df.shape}")
print(f"Unique loans     : {df['Loan Identifier'].nunique():,}")
print(f"Date range       : {df['Monthly Reporting Period'].min()} → {df['Monthly Reporting Period'].max()}")

delinq = pd.to_numeric(df["Current Loan Delinquency Status"], errors="coerce")
n_delinq_loans = df.loc[delinq > 0, "Loan Identifier"].nunique()
print(f"Ever delinquent  : {n_delinq_loans:,} loans")
print(f"Always current   : {df['Loan Identifier'].nunique() - n_delinq_loans:,} loans")

print(f"\nDelinquency distribution:")
print(df["Current Loan Delinquency Status"].value_counts().head(10))

df.head(3)

Shape            : (22374299, 30)
Unique loans     : 785,516
Date range       : 2022-01-01 00:00:00 → 2025-09-01 00:00:00
Ever delinquent  : 355,363 loans
Always current   : 430,153 loans

Delinquency distribution:
Current Loan Delinquency Status
0    20688996
1      947315
2      234836
3      115592
4       83295
5       68455
6       48518
7       37043
8       30456
9       24883
Name: count, dtype: int64


,Loan Identifier,Monthly Reporting Period,Origination Date,Channel,Original Interest Rate,Original UPB,Original Loan Term,Original Loan to Value Ratio (LTV),Original Combined Loan to Value Ratio (CLTV),Debt-To-Income (DTI),...,Current Actual UPB,Loan Age,Remaining Months to Legal Maturity,Remaining Months To Maturity,Current Loan Delinquency Status,Loan Payment History,Total Principal Current,Modification Flag,Borrower Assistance Plan,Servicing Activity Indicator
0,130357805,2022-01-01,2021-12-01,R,2.75,611000.0,360.0,68.0,68.0,42.0,...,610000.0,0.0,360.0,359.0,0,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX00,NaN,N,7,N
1,130357805,2022-02-01,2021-12-01,R,2.75,611000.0,360.0,68.0,68.0,42.0,...,610000.0,1.0,359.0,359.0,0,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX0000,0.00,N,7,N
2,130357805,2022-03-01,2021-12-01,R,2.75,611000.0,360.0,68.0,68.0,42.0,...,608000.0,2.0,358.0,358.0,0,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX000000,1095.93,N,7,N
